<a href="https://colab.research.google.com/github/janzika/MATH3261-5285/blob/main/Thermal_Wind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/janzika/MATH3261-5285/blob/387245fbd27e11a9b842e413dd6c8d784e6ea63d/data/images/ocean_currents2.jpg?raw=1" align="center"/>


# Lab 4: Thermal Wind

In this lab, we will analyse observations and discuss:

- the thermal wind equation
- inference of the across section component of ocean velocity
- relationship to sea surface height
- inference of total transport

To use this notebook on your local machine, you will need Python 3. If you don't have Python3, you can run this lab from your web browser using Google Colaboratory.

(Image reference: [UC Berkeley](https://ugc.berkeley.edu/background-content/ocean-circulation/$0))

## Background

### Thermal wind

As we saw in class, the thermal wind equations are

$$
\frac{\partial u}{\partial z} = \frac{\partial \rho}{\partial y}, \qquad \frac{\partial v}{\partial z} = - \frac{\partial \rho}{\partial x}
$$

where $u$ and $v$ are the zonal and meridional velocities and $\rho$ is density. We will use these equations to infer vertical shear and then the speed of the flow in the direction across the path taken by oceanographic voyages in different parts of the world oceans.

### Seawater density from temperature and salinity

In this lab we will calculate the density of seawater from observations of temperature and salinity. Specifically we will use the state variables 'Conservative Temperature' ($\Theta$) and Absolute Salinity ($S_A$).

Oceanographic instruments measure the conductivity, temperature and depth (via pressure, $p$) of the ocean using a device known as a CTD (after the 3 variables it measures).

Using understanding of the distribution of different salts in the global ocean, their relationship to the conductivity of sea water, and the temperaure and pressure, conductivity is converted to $S_A$. $S_A$  represents the amount of dissolved salt in water in grams of salt per gram of sea water (including the salt).

For fresh water at sea level, changes in temperature, $T$, are related to the heat content change per mass of water (one calorie of heat is required to warm 1g of water by 1K). This relationship changes depending on salinity and pressure. By calculating $\Theta$ we remove these pressure and salinity effects and return a variable who's change is related to heat.

Given $\Theta$, $S_A$ and $p$, we can calculate the desnity of sea water using a toolbox called the [Gibbs Sea Water Oceanographic Toolbox (GSW)](https://github.com/TEOS-10/GSW-Python).

### GO-SHIP Oceanographic Voyage Data

[GO-SHIP](https://go-ship.org/$0) is an international project coordinating and managing storage and access to data from oceanographic voyages around the world.

These observations are called 'sections' and involve a ship travelling in an approximately straight line (i.e. constant longitude or latitude) and taking 'casts' (vertcial profiles) measuring variables such as temperature and salinity.

GO-SHIP data is available from the Easy-Ocean data base at https://dx.doi.org/10.7942/GOSHIP-EasyOcean. This notebook contains the code to download this data for use locally or on google colab.

### Data analysis with xarray

We are going to use a different type of data structure called an xarray which is closely linked to the file format ['netcdf'](https://docs.unidata.ucar.edu/nug/current/netcdf_introduction.html) (uses the `.nc` file extension).

[xarray](https://xarray.dev/) is specially designed to deal with multidimensional arrays. An xarray data frame has it's own in-built functionality including for taking means, derivatives, multiplication, plotting and much more.

For a nice set of tutorials see for example: http://gallery.pangeo.io/repos/pangeo-data/pangeo-tutorial-gallery/xarray.html.



## Getting started and example

### Libraries and setup

Today we need a few extra libraries for gsw, netcdf and xarray.
If you are running this notebook locally you do not need to run this cell as the libraries are already installed with pixi.

In [ ]:
!pip install netcdf4
!pip install gsw
!pip install cmocean

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# import library
import netCDF4 as nc4
import gsw
import cmocean.cm as cm # for an extended range of colourmaps

# utilities to download and unzip the netcdf data
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

We also need to define a few basic varaibles to  (add more as needed)





In [ ]:
Omega = 7.2921e-5    # 1 / s
g = 9.81             # m^2 / s
rho0 = 1028          # kg / m^3
dbar2pascal = 10000

The GOSHIP data is publically available from the [GOSHIP zenodo](https://zenodo.org/records/16991635) record where the dataset is maintained. We will download and use the gridded netcdf data in this lab. The cell below can take some time as it download the whole data set.

In [ ]:
url = "https://zenodo.org/records/16991635/files/gridded_netcdf.zip"

# Download the .zip file of all GOSHIP data transects
data_dir = Path("GOSHIP")
zip_path = Path("gridded_netcdf.zip")

# Check if GOSHIP data is already downloaded, if not download, unzip and remove zip
if not data_dir.is_dir() or not any(data_dir.iterdir()):
    # Download
    urlretrieve(url, zip_path)

    # Extract
    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)

    # Remove ZIP file
    zip_path.unlink()

    print(f"Data downloaded and extracted to {data_dir}/")
else:
    print(f"Data already exists in {data_dir}/, skipping download.")

### Loading, displaying and plotting data

Let's start by choosing a section from the data folder. We are choosing SR01 because it is a section A/Prof Jan Zika has participated in making measurements for and is a nice narrow North-South section where there is a strong zonal flow.

We can load the data from a single section as a dataset (`ds`) with the `xr.open_dataset` command.

In [ ]:
SECTION_url = 'GOSHIP/gridded/SR01/sr01.nc'
ds = xr.open_dataset(SECTION_url, engine="netcdf4")

 To interogate the data we can simply type 'ds'. (Click on the headings below and try to see what variables are available, their dimensions and the length of each.)

In [ ]:
ds

To make things easy(ish), with xarray we can just choose a slice of the data and type ".plot()" at the end of the command. Here I am choosing all pressures and latitudes `[:,:]`, the 1st and only longitude `[0]` and the last section occupation `[15]`.

In [ ]:
ds.conservative_temperature[:,:,0,15].plot()

As you can see, based on the dimensions we chose to plot and the meta-data, it knows to make a heatmap and has labelled the axes and colourbar. It isn't smart enough to know pressure should increase downward so let's flip the y-axis.

In [ ]:
ds.conservative_temperature[:,:,0,15].plot(yincrease=False)

Above, we have indexed the four dimensional array containing the `conservative_temperature` variable as we would any generic array.
An advantage of the `xarray` package is that we can also [index using a named dimension](https://docs.xarray.dev/en/stable/getting-started-guide/quick-overview.html#indexing).
For example, we can produce the same plot by indexing with `isel` on the xarray object.
We also change the colour map used to `thermal` provided by the cmocean package and set custom limits for the temperature colourrange.

In [ ]:
ds.conservative_temperature.isel(longitude=0,gridded_section=15).plot(yincrease=False, cmap=cm.thermal,vmin=-1, vmax=8)

Try making an equivalent plot for salinity. Think about where salinity and temperature are making the ocean more or less stratified in density.

### Density, derivatives and multiplication

Let's employ the GSW toolbox to tell us the density and plot it. GSW isnt smart enough to give us a good name of the variable and units but we can add those. This won't change the numbers but may be useful at various stages to make sure our plots look nice and we dont loose track of the units.

In [ ]:
ds["density"] = gsw.rho(ds["absolute_salinity"], ds["conservative_temperature"], ds["pressure"])
ds["density"].attrs["units"]='kg m$^{-3}$'
ds["density"].attrs["long_name"]='sea_water_density'
ds["density"].isel(longitude=0, gridded_section=15).plot(yincrease=False, cmap=cm.dense)

Because density is such a strong function of pressure, it really doesnt look like there is any horizontal gradient at all. Thankfully, xarray has an inbuilt derivative function so we can check.

In [ ]:
ds["drhodlat|_p"] = ds.density.differentiate("latitude")
ds["drhodlat|_p"].attrs["long_name"]='density_latitude_derivative'
ds["drhodlat|_p"].attrs["units"]='kg m$^{-3} deg^{-1}$'
ds["drhodlat|_p"].isel(longitude=0, gridded_section=15).plot(yincrease=False, cmap=cm.balance)

Think about what the above might say about the velocity structure across the section.

### Integration and the height above the bottom

We can consider the hydrostatic equation to see if it might be able to tell us what the height is as a function of pressure and density.
We have

$$ \frac{\partial p}{\partial z} = -g\rho.$$

Inverting and integrating with respect to $p$ we have

$$ z - \eta = \int_{p_b}^{p}\frac{\partial z}{\partial p}dp = \int_{p_b}^{p}-\frac{1}{g\rho} dp.$$

Above, $z_b$ and $p_b$ is the depth and pressure at the bottom respectively. We can calculate this integral using the `cumsum` function and multiplying by the $\Delta p$ of layers.

In [ ]:
dp = 10*dbar2pascal #data is in 10 decibar increments but correct SI unit is pascal
ds["z_rel_eta"] = -(1/(g*ds["density"][:,::-1,:,:])).cumsum("pressure")[:,::-1,:,:]*dp
ds["z_rel_eta"].attrs["units"]='m'
ds["z_rel_eta"].attrs["long_name"]='height_relative_surface'
ds["z_rel_eta"] = ds["z_rel_eta"].where(ds["temperature"].notnull())
ds["z_rel_eta"].isel(longitude=0, gridded_section=15).plot(x="latitude",y="pressure",yincrease=False)


Above, we added one line to make the solid earth a NaN (white space in the plot) the same way it is in the other variables.

You can see that the height is very close to the -p in dbar units. It is common in oceanography to apporximate depth as simply

$$p = p_{atm} + \rho_0 g h.$$

## Your Tasks



**Theory**

(i) Start with the equation for hydrostatic balance
$$
\frac{dp}{dz} = - \rho_0 g.
$$

Integrate this equation from depth $z<0$ to the sea surface $z = \eta(x,y)$, taking the atmospheric pressure to be $p_0 = $ constant and the pressure at depth $z$ to be $p(x,y,z)$. Using the resulting expression, show that the geostrophic current at the surface is
$$
u^\mathrm{surf} (x,y) = - \frac{g}{f_0} \frac{\partial \eta}{\partial y}, \qquad
v^\mathrm{surf} (x,y) = \frac{g}{f_0} \frac{\partial \eta}{\partial x}.
$$
(You can take $\rho_0$, $f_0$ as constants.) Find an expression for the geostrophic streamfunction $\psi (x,y)$ in terms of the sea-surface height $\eta(x,y)$.

(ii) An oceanographic vessel has travelled along a line and taken measurements (`casts') of temperature and salinity vertically at regular points along its track. The measurements are converted to density using the equation of state for sea water. Given density as a function of depth and along track distance, integrate the thermal wind equation, to find an expression for the across track geostrophic currents.

**Lab Component**

Choose a unique section from those provided on the class github repo.

By assuming the velocity is zero at the bottom calculate the implied velocity as a function of depth.

Determine the implied velocty at the sea surface and what the hight of the sea surface should be relative to it's height at the coast given your implied velocity.

Work as a class to map the strength and direction of the major ocean currents of global ocean.

